In [0]:
# CALIDAD DE DATOS — generamos el reporte formal de calidad
# mostramos cuantos registros pasan y fallan por cada regla


import time
from pyspark.sql import functions as f
from pyspark.sql import Row

CATALOGO     = "nyc_taxi_andres"
CAPA_RAW     = f"{CATALOGO}.raw"
CAPA_TRUSTED = f"{CATALOGO}.trusted"
CAPA_REFINED = f"{CATALOGO}.refined"

# estos numeros los tenemos del notebook de trusted
# los definimos aqui para construir el reporte formal
TOTAL_INICIAL = 3_066_766

descartados = {
    "tiempo_invalido"     : 1121,
    "sin_distancia"       : 44814,
    "tarifa_invalida"     : 22522,
    "outliers_extremos"   : 3046,
    "nulos_columnas_clave": 0
}

print("Configuracion lista")

In [0]:
# en lugar de usar solo los numeros guardados,
# validamos  contra la tabla raw para que el reporte
# sea dinamico y no dependa de valores quemados

print("Validando reglas contra la tabla raw...")
inicio = time.time()

viajes_raw = spark.table(f"{CAPA_RAW}.viajes_enero_2023")

# aplicamos cada regla y contamos pasan/fallan
reglas = []

# regla 1: tiempo valido
fallan = viajes_raw.filter(
    f.col("tpep_dropoff_datetime") <= f.col("tpep_pickup_datetime")
).count()
pasan = TOTAL_INICIAL - fallan
reglas.append(Row(
    regla="tiempo_valido",
    descripcion="pickup debe ser menor que dropoff",
    registros_pasan=pasan,
    registros_fallan=fallan,
    porcentaje_falla=round((fallan / TOTAL_INICIAL) * 100, 2)
))
print(f"Regla 1 validada: {fallan:,} fallan")

# regla 2: distancia positiva
fallan = viajes_raw.filter(f.col("trip_distance") <= 0).count()
pasan = TOTAL_INICIAL - fallan
reglas.append(Row(
    regla="distancia_positiva",
    descripcion="trip_distance debe ser mayor a cero",
    registros_pasan=pasan,
    registros_fallan=fallan,
    porcentaje_falla=round((fallan / TOTAL_INICIAL) * 100, 2)
))
print(f"Regla 2 validada: {fallan:,} fallan")

# regla 3: tarifa positiva
fallan = viajes_raw.filter(f.col("fare_amount") <= 0).count()
pasan = TOTAL_INICIAL - fallan
reglas.append(Row(
    regla="tarifa_positiva",
    descripcion="fare_amount debe ser mayor a cero",
    registros_pasan=pasan,
    registros_fallan=fallan,
    porcentaje_falla=round((fallan / TOTAL_INICIAL) * 100, 2)
))
print(f"Regla 3 validada: {fallan:,} fallan")

# regla 4: sin nulos en columnas clave
fallan = viajes_raw.filter(
    f.col("PULocationID").isNull() |
    f.col("tpep_pickup_datetime").isNull() |
    f.col("fare_amount").isNull()
).count()
pasan = TOTAL_INICIAL - fallan
reglas.append(Row(
    regla="sin_nulos_columnas_clave",
    descripcion="PULocationID, pickup_datetime y fare_amount no pueden ser nulos",
    registros_pasan=pasan,
    registros_fallan=fallan,
    porcentaje_falla=round((fallan / TOTAL_INICIAL) * 100, 2)
))
print(f"Regla 4 validada: {fallan:,} fallan")

print(f"\nValidacion completada en {round(time.time() - inicio, 2)}s")

In [0]:
# construimos el dataframe y lo guardamos en refined
# esta es la tabla que se pide para la prueba

reporte = spark.createDataFrame(reglas)

reporte.write.format("delta").mode("overwrite").saveAsTable(
    f"{CAPA_REFINED}.data_quality_report"
)

print("Reporte de calidad guardado en refined.data_quality_report")
print("\n resultado del reporte")
reporte.show(truncate=False)

In [0]:
print("  REPORTE DE CALIDAD DE DATOS")
print("=" * 55)
print(f"  Total registros evaluados : {TOTAL_INICIAL:,}")
print(f"  Reglas aplicadas          : {len(reglas)}")
print("")

for r in reglas:
    estado = "OK" if r.registros_fallan == 0 else "FALLA"
    print(f"  [{estado}] {r.regla}")
    print(f"         Pasan  : {r.registros_pasan:,}")
    print(f"         Fallan : {r.registros_fallan:,} ({r.porcentaje_falla}%)")
    print("")

print("=" * 55)
print("  Tabla guardada: refined.data_quality_report")
print("=" * 55)